# CAD-AI Inspector — Option A (Pretrained AI Only)

**Grounding DINO Tiny + DINOv2 Small · Kaggle GPU · No training or fine-tuning**

This notebook audits the supplied dataset, downloads public pretrained checkpoints, and runs inference over all real-photo views. Predictions are not accuracy metrics because per-part ground-truth labels are not included.

In [1]:
!nvidia-smi
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

Wed Nov  4 21:30:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 592.00                 Driver Version: 592.00         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   56C    P8              6W /   50W |     218MiB /   4096MiB |     13%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Change this path if you extracted the project elsewhere.

!pip install -r requirements.txt

     ---------------------------------------- 43.8/43.8 MB 3.6 MB/s eta 0:00:00
     ---------------------------------------- 12.8/12.8 MB 3.6 MB/s eta 0:00:00
     -------------------------------------- 394.3/394.3 kB 3.5 MB/s eta 0:00:00
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
     -------------------------------------- 123.3/123.3 kB 3.6 MB/s eta 0:00:00
     ---------------------------------------- 2.9/2.9 MB 4.1 MB/s eta 0:00:00
     -------------------------------------- 355.5/355.5 kB 5.6 MB/s eta 0:00:00
     -------------------------------------- 857.0/857.0 kB 5.4 MB/s eta 0:00:00
     -------------------------------------- 280.6/280.6 kB 2.9 MB/s eta 0:00:00
     ---------------------------------------- 3.9/3.9 MB 2.5 MB/s eta 0:00:00
     ---------------------------------------- 96.9/96.9 kB 2.8 MB/s eta 0:00:00
     -------------------------------------- 133.0/133.0 kB 4.0 MB/s eta 0:00:

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\tanis\\anaconda3\\Lib\\site-packages\\safetensors\\_safetensors_rust.pyd'
Consider using the `--user` option or check the permissions.



  Using cached opencv_python_headless-5.0.0.93-cp37-abi3-win_amd64.whl (43.8 MB)
  Using cached transformers-5.20.2-py3-none-any.whl (12.8 MB)
  Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl (355 kB)
     -------------------------------------- 857.0/857.0 kB 2.3 MB/s eta 0:00:00
  Using cached regex-2026.9.29-cp310-cp310-win_amd64.whl (280 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl (2.9 MB)
  Using cached typer-0.27.3-py3-none-any.whl (123 kB)
     ---------------------------------------- 3.9/3.9 MB 2.1 MB/s eta 0:00:00
  Using cached click-8.5.0-py3-none-any.whl (125 kB)
  Using cached httpx2-2.14.0-py3-none-any.whl (96 kB)
  Using cached filelock-4.1.0-py3-none-any.whl (133 kB)
  Using cached anyio-4.16.0-py3-none-any.whl (134 kB)
  Using cached truststore-0.10.5-py3-none-

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\tanis\\anaconda3\\Lib\\site-packages\\~umpy.libs\\libopenblas64__v0.3.23-293-gc2f4bdbb-gcc_10_3_0-2bde3a66a51006b2b53eb373ff767a3f.dll'
Consider using the `--user` option or check the permissions.



## 1. Audit the supplied data (no pretrained models needed)

In [4]:
!python scripts/dataset_audit.py --data data/sample/AI_sample_CAD --output outputs/dataset_audit
from IPython.display import Markdown, display, Image
print(open('outputs/dataset_audit/DATASET_AUDIT.md').read())

# Supplied dataset audit

> This is an actual dataset-structure and mask-integrity audit. It is not a pretrained-model inference result.

## AI_sample_CAD
- JSON components: 32
- Mask dimensions: 4032 × 3024
- Distinct red-channel mask values: 30
- Components with nonzero mask pixels: 29
- Components with zero mask pixels: 3
- CAD dimensions: [3024, 4032, 3]
- Real captures:
  - BG_00.png: 4032 × 3024 (3 channels)
  - BG_00_TestImage.png: 4032 × 3024 (3 channels)
  - BG_00_TestImage2.png: 4032 × 3024 (3 channels)
  - BG_00_TestImage3.png: 4032 × 3024 (3 channels)

## TestCase2
- JSON components: 32
- Mask dimensions: 2902 × 2176
- Distinct red-channel mask values: 23
- Components with nonzero mask pixels: 22
- Components with zero mask pixels: 10
- CAD dimensions: [2176, 2902, 3]
- Real captures:
  - BG_00.png: 2902 × 2176 (3 channels)




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\tanis\Downloads\CAD_AI_Inspection_OptionA_Kaggle\scripts\dataset_audit.py", line 5, in <module>
    import cv2, numpy as np
  File "c:\Users\tanis\anaconda3\lib\site-packages\cv2\__init__.py", line 181, in <module>
    bootstrap()
  File "c:\Users\tanis\anaconda3\lib\site-packages\cv2\__init__.py", line 153, in bootstrap
    native_module = importlib.import_module("cv2")
  File "c:\Users\tanis\anaconda3\lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], p

## 2. Download pretrained models

The weights are not bundled in the project. Internet access must be enabled in Kaggle. Both selected checkpoints are public; an HF token is optional.

In [ ]:
!python scripts/download_models.py --models grounding-dino dinov2

## 3. Run inference over every supplied real-image view

This loads the two models once and evaluates the four TestCase1 photographs plus the TestCase2 photograph. It writes one folder per view, an aggregate CSV and a view summary.

In [ ]:
!python scripts/run_all_views.py --data data/sample/AI_sample_CAD --output outputs/full_supplied_experiment --device auto

## 4. Review the results

In [ ]:
import pandas as pd, glob, os
summary = pd.read_csv('outputs/full_supplied_experiment/view_summary.csv')
summary

In [ ]:
predictions = pd.read_csv('outputs/full_supplied_experiment/all_predictions.csv')
predictions.head(40)

In [ ]:
# Display one annotated view
files = sorted(glob.glob('outputs/full_supplied_experiment/*/annotated_result.png'))
print(files[0] if files else 'No annotated results generated')
if files: display(Image(filename=files[0]))

## 5. Interpretation

- `PRESENT` is a Grounding-DINO proposal that also passes the configured DINOv2 CAD-reference similarity threshold.
- `UNCERTAIN` means the system cannot confidently establish presence or absence.
- `ABSENT` requires a verified real-image expected ROI with `visible: true`; simple detector non-detection is never enough.

There are no supplied per-view component ground-truth labels, so do not report precision/recall/accuracy from these predictions. To evaluate accuracy, annotate a holdout set and calculate per-component metrics.